# Interpolations

L’interpolation est une technique mathématique qui permet d’estimer des valeurs manquantes ou intermédiaires à partir d’un ensemble de points connus. L’idée est de construire une fonction qui relie ces points, de façon à pouvoir calculer des valeurs pour n’importe quel point situé entre eux.

## Imports & Variables

In [ ]:
import os
import sys
sys.path.append(os.path.abspath("../../../"))

from src.methodes import *
from src.visualisations import *
from src.data import *

In [ ]:
valeur_de_travail = 'T_Q'

fichier_nappe = "../../../data/fusion/data_03288X0042_P.csv"
dossier_nappe = "../../../data/fusion"

df = charger_fichier(fichier_nappe)

## Interpolation linéaire

L’interpolation linéaire permet d’estimer des valeurs manquantes en supposant que l’évolution entre deux points connus suit une droite. Les valeurs manquantes sont donc calculées à partir de la droite reliant les observations voisine

$$
f(x) = ax + b
$$

où :

- $a$ représente la pente 
- $b$ l’ordonnée à l’origine

In [ ]:
affiche_prediction(df, interpolation_lineaire_array(df, valeur_de_travail), valeur_de_travail)

## Interpolation via spline cubique

L’interpolation cubique estime les valeurs manquantes en utilisant un polynôme de degré 3 entre les points connus. Contrairement à l’interpolation linéaire, cette méthode permet d’obtenir une courbe plus lisse et de mieux capturer les variations non linéaires des données.

$$
f(x) = ax^3 + bx^2 + cx + d
$$

où :

- $a$, $b$, $c$ et $d$ sont des coefficients déterminés à partir des points connus afin d'ajuster le polynôme aux données.

In [ ]:
affiche_prediction(df, interpolation_cubique_array(df, valeur_de_travail), valeur_de_travail)

## Interpolation via Polynômes de degré n

L’interpolation polynomiale estime les valeurs manquantes en utilisant un polynôme de degré $n$ ajusté aux points connus.

$$
f(x) = a_n x^n + a_{n-1} x^{n-1} + \dots + a_1 x + a_0
$$

où :

- $n$ est le degré du polynôme,
- $a_0, a_1, \dots, a_n$ sont des coefficients déterminés à partir des points connus.

In [ ]:
affiche_prediction(df, interpolation_polynomiale_array(df, valeur_de_travail), valeur_de_travail)

## Interpolation via B-spline

L’interpolation par spline estime les valeurs manquantes en utilisant des polynômes de degré $n$ définis sur plusieurs intervalles entre les points connus. Cela permet d’obtenir une courbe plus lisse et plus stable qu’une interpolation polynomiale globale.

$$
f(x) = ax^3 + bx^2 + cx + d
$$

où :

- $a$, $b$, $c$ et $d$ sont des coefficients déterminés à partir des points connus.

In [ ]:
affiche_prediction(df, interpolation_spline_array(df, valeur_de_travail), valeur_de_travail)

## Interpolation via PCHIP

L’interpolation PCHIP (Piecewise Cubic Hermite Interpolating Polynomial) utilise des polynômes cubiques **par morceaux** pour estimer les valeurs manquantes, en préservant la **monotonie** et les **formes locales** de la série.  

$$
f(x) = S_i(x) = h_{00}(t) y_i + h_{10}(t) (x_{i+1}-x_i) m_i + h_{01}(t) y_{i+1} + h_{11}(t) (x_{i+1}-x_i) m_{i+1}, \quad x \in [x_i, x_{i+1}]
$$

avec :

$$
t = \frac{x - x_i}{x_{i+1} - x_i}
$$

$$
h_{00}(t) = 2t^3 - 3t^2 + 1
$$

$$
h_{10}(t) = t^3 - 2t^2 + t
$$

$$
h_{01}(t) = -2t^3 + 3t^2
$$

$$
h_{11}(t) = t^3 - t^2
$$

où :

- $y_i = f(x_i)$ : valeur au point $x_i$
- $y_{i+1} = f(x_{i+1})$
- $m_i, m_{i+1}$ : pentes (dérivées) aux points $x_i$ et $x_{i+1}$
- $t$ : variable normalisée dans $[0,1]$

In [ ]:
affiche_prediction(df, interpolation_pchip_array(df, valeur_de_travail), valeur_de_travail)

## Interpolation via akima

L’interpolation Akima utilise également des polynômes cubiques **par morceaux**, mais avec un calcul spécial des pentes pour éviter les oscillations excessives. Elle est utile pour les données présentant des **variations abruptes**.  

$$
f(x) = A_i(x) = h_{00}(t) y_i + h_{10}(t) (x_{i+1}-x_i) m_i + h_{01}(t) y_{i+1} + h_{11}(t) (x_{i+1}-x_i) m_{i+1}, \quad x \in [x_i, x_{i+1}]
$$

avec :

$$
t = \frac{x - x_i}{x_{i+1} - x_i}
$$

$$
h_{00}(t) = 2t^3 - 3t^2 + 1
$$

$$
h_{10}(t) = t^3 - 2t^2 + t
$$

$$
h_{01}(t) = -2t^3 + 3t^2
$$

$$
h_{11}(t) = t^3 - t^2
$$


On définit d’abord les pentes entre points :

$$
\Delta_i = \frac{y_{i+1} - y_i}{x_{i+1} - x_i}
$$

Puis les poids :

$$
w_i = |\Delta_{i+1} - \Delta_i|
$$

Les dérivées aux points sont alors données par :

$$
m_i =
\begin{cases}
\frac{w_{i+1} \Delta_{i-1} + w_{i-1} \Delta_i}{w_{i+1} + w_{i-1}} & \text{si } w_{i+1} + w_{i-1} \neq 0 \\
\frac{\Delta_{i-1} + \Delta_i}{2} & \text{sinon}
\end{cases}
$$

où :
- $y_i = f(x_i)$ : valeur au point $x_i$
- $y_{i+1} = f(x_{i+1})$
- $t$ : variable normalisée dans $[0,1]$

ubique sur l’intervalle $[x_i, x_{i+1}]$ avec des pentes calculées selon la méthode d’Akima¹.

méthode d'Akima¹ : Les pentes aux points intermédiaires ne sont pas simplement calculées par la pente linéaire des segments voisins, mais selon un moyen pondéré des pentes locales des segments adjacents 

In [1]:
affiche_prediction(df, interpolation_akima_array(df, valeur_de_travail), valeur_de_travail)

NameError: name 'affiche_prediction' is not defined